# Section III/IV: Market Simulation and Stylized Facts

**Talk section**: Section III (simulation and results) / Section IV (open problems setup)

This is the hero notebook — it contains the central empirical result of the project. We run three simulation conditions, analyze the results for stylized facts, and demonstrate that **only the mixed-population market reproduces the statistical properties of real financial markets**.

By the end of this notebook you will have:
- Run the three hero experiment conditions (momentum-only, value-only, mixed)
- Computed log returns and return distribution statistics for each condition
- Plotted return distributions with normal overlays (showing fat tails)
- Computed and plotted ACF of squared returns (showing volatility clustering)
- Saved the results to `eval/results/hero_experiment.json` for the Streamlit app
- Seen why these results set up Section IV's open problems

## The Market Simulation Design

The simulation is intentionally minimal. The goal is to isolate the behavioral signal — the effect of persona type on market dynamics — not to build a realistic market microstructure.

**Key design choices:**

1. **One risky asset**: No portfolio effects, no hedging, no cross-asset correlation. One price series.

2. **Price update rule**:
   ```
   price_{t+1} = price_t * (1 + alpha * order_imbalance_t + epsilon_t)
   ```
   - `order_imbalance = (buy_volume - sell_volume) / total_volume` — the aggregate behavioral signal
   - `alpha = 0.01` — price impact coefficient (how much imbalance moves the price)
   - `epsilon ~ N(0, 0.001)` — exogenous noise (random shocks not explained by order flow)

3. **No short selling, no leverage, no transaction costs**: Agents can only buy shares they have cash for and sell shares they hold. This keeps the simulation self-contained and avoids degenerate behavior.

4. **Rule-based agents**: We use `RuleBasedAgent` which implements each persona's logic as a simple hand-coded rule. No GPU, no model weights required. The stylized facts emerge from the behavioral rules alone.

The simplifications are deliberate. A production agent-based model would include order books, partial fills, position limits, risk management, etc. Here, we want the cleanest possible demonstration that *behavioral heterogeneity, not microstructural complexity, generates stylized facts*.

In [ ]:
# Imports
import json
import sys
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats

# Set matplotlib style for a clean dark-theme look
matplotlib.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#0E1117'
plt.rcParams['axes.facecolor'] = '#1A1F2E'
plt.rcParams['grid.alpha'] = 0.2

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from simulation.market import Market
from simulation.agent import RuleBasedAgent
from simulation.run_simulation import run_simulation, create_agents
from eval.stylized_facts import compute_log_returns, test_fat_tails, test_volatility_clustering

print("All imports successful.")
print(f"Project root: {project_root}")

## Agent Types

We use `RuleBasedAgent` which runs on CPU without any model weights. The three persona rules are:

**Momentum**: If the last 3 prices are strictly increasing → BUY. If strictly decreasing → SELL. Otherwise → HOLD. Quantity is proportional to trend strength.

**Value**: If current price < 95% of fair value → BUY. If > 105% of fair value → SELL. Otherwise → HOLD. Quantity is proportional to deviation from fair value.

**Noise**: 40% probability of HOLD. Otherwise, 55% probability of following the recent price direction (trend-chasing) and 45% of going against it. Random quantity in [1, 20].

These rules are simple but they capture the essence of each persona. The hero experiment result does not depend on the rules being complex — it depends on them being *different*.

In [ ]:
# Demonstrate the agent decision logic with a concrete example

example_market_state = {
    "price_history": [100.0, 102.0, 105.0, 107.0, 110.0],
    "current_price": 110.0,
    "fair_value": 104.0,
    "news": "Strong earnings report.",
    "tick": 5,
}

print(f"Market state: price={example_market_state['current_price']}, fair_value={example_market_state['fair_value']}")
print(f"Price history: {example_market_state['price_history']}")
print()

for persona in ["momentum", "value", "noise"]:
    # Use a fixed seed for the noise trader so output is reproducible
    agent = RuleBasedAgent(persona, f"{persona}_demo", seed=42)
    decision = agent.decide(example_market_state)
    print(f"  [{persona.upper()}] -> {decision['action']} {decision['quantity']} shares")

## The Three Experimental Conditions

The hero experiment runs three simulation conditions, each with 30 agents and 500 ticks:

1. **Homogeneous Momentum**: 30 momentum traders. They all follow the same trend-chasing logic. In a trending market, they all buy simultaneously, amplifying the trend. When the trend reverses, they all sell. Expected result: persistent trends, herding, little mean-reversion.

2. **Homogeneous Value**: 30 value investors. They all anchor to the same fair value. When the price deviates, they collectively push it back. Expected result: stable prices, rapid mean-reversion, low volatility.

3. **Mixed Population**: 10 momentum + 10 value + 10 noise traders. Momentum traders amplify trends; value traders push back; noise traders add random bursts. The interaction between these opposing forces creates non-linear dynamics. Expected result: fat tails, volatility clustering.

In [ ]:
# Run the three hero experiment conditions
# Each run takes a few seconds on CPU

N_TICKS = 500
SEED = 42
INITIAL_PRICE = 100.0
FAIR_VALUE = 100.0

conditions = {
    "momentum_only": {"momentum": 30},
    "value_only": {"value": 30},
    "mixed": {"momentum": 10, "value": 10, "noise": 10},
}

condition_names = {
    "momentum_only": "Homogeneous Momentum",
    "value_only": "Homogeneous Value",
    "mixed": "Mixed Population",
}

results = {}

for label, persona_counts in conditions.items():
    print(f"Running: {condition_names[label]} ({persona_counts})...")
    agents = create_agents(persona_counts, seed_offset=SEED)
    log = run_simulation(
        agents,
        n_ticks=N_TICKS,
        initial_price=INITIAL_PRICE,
        fair_value=FAIR_VALUE,
        seed=SEED,
    )
    results[label] = log
    summary = log["market_summary"]
    print(f"  Done. Final price: {summary['final_price']:.2f} "
          f"(started at {summary['initial_price']:.2f})")

print("\nAll simulations complete.")

## What Are Stylized Facts?

Stylized facts are empirical regularities that appear consistently across financial markets, asset classes, and time periods. They are the empirical benchmark that any model of financial markets must reproduce.

The two most robust stylized facts in the literature (Cont 2001) are:

**1. Fat Tails (Leptokurtosis)**: Real return distributions have heavier tails than a normal distribution. Large moves — crashes, rallies — occur far more often than Gaussian models predict. Formally measured by **kurtosis > 3** (equivalently, **excess kurtosis > 0**).

**2. Volatility Clustering**: The magnitude of price moves is autocorrelated. A large move (in either direction) tends to be followed by another large move. Formally measured by **positive autocorrelation of squared returns** at short lags. This is the empirical basis for GARCH models.

A homogeneous momentum-trader market produces trending prices but returns that are roughly normal (all agents react similarly, creating smooth trends with similar-sized moves). A homogeneous value-trader market produces near-constant prices with very low volatility. Neither reproduces the stylized facts.

The mixed market, with opposing behavioral forces, generates the non-linear dynamics needed for both properties.

In [ ]:
# Compute log returns and basic statistics for each condition

condition_stats = {}

for label, log in results.items():
    returns = compute_log_returns(log["price_series"])
    fat_tails = test_fat_tails(returns)
    clustering = test_volatility_clustering(returns, max_lag=20)
    
    condition_stats[label] = {
        "returns": returns,
        "fat_tails": fat_tails,
        "clustering": clustering,
    }
    
    print(f"\n=== {condition_names[label]} ===")
    print(f"  N returns: {fat_tails['n_returns']}")
    print(f"  Excess kurtosis: {fat_tails['excess_kurtosis']:.3f} "
          f"({'FAT TAILS' if fat_tails['is_fat_tailed'] else 'near-normal'})")
    print(f"  JB p-value: {fat_tails['jb_pvalue']:.4f} "
          f"({'reject normality' if fat_tails['jb_pvalue'] < 0.05 else 'cannot reject normality'})")
    print(f"  Mean ACF (lags 1-5): {clustering['mean_acf_lags_1_5']:.4f} "
          f"({'CLUSTERING' if clustering['has_clustering'] else 'no clustering'})")
    print(f"  LB p-value: {clustering['lb_pvalue']:.4f}")

## Return Distribution Plots

Each plot shows the histogram of log returns for one condition, with a fitted normal distribution overlaid in white. Fat tails manifest as the histogram having more mass in the extreme tails (far from zero) than the normal curve.

The excess kurtosis annotation in each panel summarizes the deviation from normality. Normal distribution has excess kurtosis = 0. Positive values indicate fat tails.

In [ ]:
# Plot return distributions side by side

condition_colors = {
    "momentum_only": "#E8A838",
    "value_only": "#3A86FF",
    "mixed": "#FF6B6B",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Return Distributions: Do we get fat tails?", fontsize=13, fontweight='bold')

for ax, (label, stat_data) in zip(axes, condition_stats.items()):
    returns = stat_data["returns"]
    color = condition_colors[label]
    name = condition_names[label]
    fat_tails = stat_data["fat_tails"]
    
    # Histogram
    ax.hist(returns, bins=40, density=True, alpha=0.65, color=color, edgecolor='none')
    
    # Normal overlay
    mu, sigma = np.mean(returns), np.std(returns)
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'w--', linewidth=1.5, alpha=0.85, label='Normal fit')
    
    # Excess kurtosis annotation
    ek = fat_tails["excess_kurtosis"]
    fat_label = "Fat tails" if fat_tails["is_fat_tailed"] else "Near-normal"
    ax.text(0.97, 0.95, f"Excess kurtosis: {ek:.2f}\n{fat_label}",
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#1A1F2E', edgecolor=color, alpha=0.8))
    
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Log Return")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(project_root / "eval" / "results" / "return_distributions.png", 
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

## Volatility Clustering

Each plot shows the autocorrelation function (ACF) of squared returns at lags 1 through 20. The white dashed lines are the 95% confidence bands for zero autocorrelation (±1.96/√N).

Bars that exceed the confidence band indicate statistically significant autocorrelation at that lag. A pattern of multiple significant positive bars at short lags is the signature of volatility clustering.

For the mixed population, we expect to see bars persistently above the confidence band at short lags (1-5), indicating that large moves tend to follow large moves — the volatility clustering stylized fact.

In [ ]:
# Plot ACF of squared returns (volatility clustering test)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("ACF of Squared Returns: Do we get volatility clustering?", fontsize=13, fontweight='bold')

for ax, (label, stat_data) in zip(axes, condition_stats.items()):
    returns = stat_data["returns"]
    color = condition_colors[label]
    name = condition_names[label]
    clustering = stat_data["clustering"]
    
    lags = clustering["lags"]
    acf_vals = clustering["acf_squared_returns"]
    n_obs = clustering["n_returns"]
    conf = 1.96 / math.sqrt(max(n_obs, 1))
    
    if lags:
        bar_colors = [color if v > conf else '#555555' for v in acf_vals]
        ax.bar(lags, acf_vals, color=bar_colors, alpha=0.8, width=0.7)
        ax.axhline(y=conf, color='white', linestyle='--', linewidth=1, alpha=0.6)
        ax.axhline(y=-conf, color='white', linestyle='--', linewidth=1, alpha=0.6)
        ax.axhline(y=0, color='white', linewidth=0.5, alpha=0.3)
        
        has_clust = clustering["has_clustering"]
        mean_acf = clustering["mean_acf_lags_1_5"]
        cluster_label = "CLUSTERING DETECTED" if has_clust else "No clustering"
        cluster_color = "#50FA7B" if has_clust else "#FF6B6B"
        ax.text(0.97, 0.95, f"{cluster_label}\nMean ACF (1-5): {mean_acf:.3f}",
                transform=ax.transAxes, ha='right', va='top', fontsize=9,
                color=cluster_color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#1A1F2E', edgecolor=cluster_color, alpha=0.8))
    
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Lag")
    ax.set_ylabel("Autocorrelation")

plt.tight_layout()
plt.savefig(project_root / "eval" / "results" / "volatility_clustering.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

## Interpreting the Results

**What we expect to see:**

- **Homogeneous Momentum**: Low excess kurtosis (momentum traders all move in the same direction at the same time, creating large but smooth trends rather than erratic spikes). Low volatility clustering (moves are directional, not heteroskedastic).

- **Homogeneous Value**: Very low excess kurtosis (value investors keep the price near fair value; returns are small and nearly symmetric). Very low volatility clustering (the market is near-stationary).

- **Mixed Population**: Higher excess kurtosis and positive short-lag ACF of squared returns. The conflict between momentum amplification and value mean-reversion creates occasional large moves followed by further large moves — the stylized facts signature.

**Why this matters for Section IV:**

These results are produced by rule-based agents with no memory and no context. Now ask: what happens when you replace the rule-based agents with SLM agents whose behavior is conditioned on a growing context window? The rule-based agents have perfectly stable personas by construction. The SLM agents do not — their behavior may drift as the context accumulates. That is the context drift problem (Section IV, `eval/persona_drift.py`).

In [ ]:
# Kurtosis comparison: the money shot
# Side-by-side bar chart showing excess kurtosis for all three conditions

fig, ax = plt.subplots(figsize=(8, 4))
fig.suptitle("Excess Kurtosis by Condition: The Money Shot", fontsize=13, fontweight='bold')

labels = [condition_names[k] for k in conditions.keys()]
kurtosis_vals = [condition_stats[k]["fat_tails"]["excess_kurtosis"] for k in conditions.keys()]
colors = [condition_colors[k] for k in conditions.keys()]

bars = ax.bar(labels, kurtosis_vals, color=colors, alpha=0.8, width=0.5)

# Normal distribution reference line
ax.axhline(y=0, color='white', linestyle='--', linewidth=1.5, alpha=0.6, 
           label='Normal distribution (excess kurtosis = 0)')

# Value labels on bars
for bar, val in zip(bars, kurtosis_vals):
    y_pos = bar.get_height() + 0.05 if val >= 0 else bar.get_height() - 0.15
    ax.text(bar.get_x() + bar.get_width()/2, y_pos,
            f"{val:.2f}", ha='center', va='bottom',
            fontsize=12, fontweight='bold', color='white')

ax.set_ylabel("Excess Kurtosis", fontsize=11)
ax.legend(fontsize=9)
ax.set_title("Only the mixed population shows fat tails", fontsize=10, style='italic')

plt.tight_layout()
plt.savefig(project_root / "eval" / "results" / "kurtosis_comparison.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

In [ ]:
# Save hero experiment results to eval/results/hero_experiment.json
# This file powers the Hero Experiment page in the Streamlit app

output_dir = project_root / "eval" / "results"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "hero_experiment.json"

hero_output = {}

for label, log in results.items():
    hero_output[label] = {
        "name": condition_names[label],
        "persona_composition": log["persona_composition"],
        "price_series": log["price_series"],
        "order_flow": log["order_flow"],
        "market_summary": log["market_summary"],
        "metadata": log["metadata"],
    }

with open(output_path, "w") as f:
    json.dump(hero_output, f, indent=2)

print(f"Hero experiment results saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")
print()
print("This file is used by:")
print("  - app/pages/03_hero_experiment.py (Streamlit app)")
print("  - eval/stylized_facts.py run_hero_analysis()")

## Conclusion and Open Problems Teaser

We have demonstrated the central claim of this project:

> **Behavioral heterogeneity at the micro level (three distinct trader personas) produces statistical regularities at the macro level (fat tails, volatility clustering) that homogeneous populations cannot replicate.**

This result holds even with the simplest possible rules for each persona. The stylized facts are not an artifact of model complexity — they emerge from the *conflict* between behavioral types.

---

**This sets up three open problems for Section IV:**

1. **Context drift** (`eval/persona_drift.py`): Our rule-based agents have perfectly stable personas by construction. SLM agents do not — as their context window fills with market history, their behavior may drift. How do we measure and control for this?

2. **Persona identifiability**: Can we recover an agent's persona type from its trade history alone? This is the inverse problem. It matters for model validation and for applying this framework to real market data.

3. **Equilibrium and stability**: Under what persona compositions does the market remain stationary? Is there a critical fraction of value investors below which the market becomes non-stationary? Can we characterize this analytically?

These are not rhetorical questions — they are active research directions. The stubs in `eval/persona_drift.py` outline the measurement frameworks needed to start answering them.